In [1]:
import numpy as np, xgboost as xgb, joblib, os, shutil
from collections import Counter
os.makedirs('../models', exist_ok=True)
x_train_b = np.load('../data/processed/x_train.npy')
x_test_b  = np.load('../data/processed/x_test.npy')
y_train_b = np.load('../data/processed/y_train_binary.npy')
y_test_b  = np.load('../data/processed/y_test_binary.npy')
x_train_m = np.load('../data/processed/x_train.npy')
x_test_m  = np.load('../data/processed/x_test.npy')
y_train_m = np.load('../data/processed/y_train_multi.npy')
y_test_m  = np.load('../data/processed/y_test_multi.npy')
print(f"x_train_b: {x_train_b.shape}")
print(f"y_train_b: {y_train_b.shape}")
print(f"y_train_m: {y_train_m.shape}")
print(" Données chargées")

x_train_b: (131926, 134)
y_train_b: (131926,)
y_train_m: (131926,)
 Données chargées


In [2]:
# MODÈLE 1 BINAIRE - Hyperparamètres légèrement optimisés
params_binary = {
    'max_depth': 5,           # 4 → 5 (un peu plus expressif)
    'eta': 0.05,
    'gamma': 0.3,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_lambda': 8,          # 10 → 8 (un peu moins régularisé)
    'alpha': 0.1,
    'min_child_weight': 8,    # 10 → 8
    'scale_pos_weight': 2,
    'tree_method': 'hist',
    'objective': 'binary:logistic',
    'verbosity': 1
}

dtrain_b = xgb.DMatrix(x_train_b, label=y_train_b)
dtest_b  = xgb.DMatrix(x_test_b,  label=y_test_b)

evals_b = [(dtrain_b,'train'), (dtest_b,'test')]

model_binary = xgb.train(
    params_binary, dtrain_b,
    num_boost_round=500,
    evals=evals_b,
    early_stopping_rounds=30,
    verbose_eval=50
)
print("Modèle 1 entraîné")

[0]	train-logloss:0.69341	test-logloss:0.65859
[50]	train-logloss:0.06148	test-logloss:0.41835
[58]	train-logloss:0.04765	test-logloss:0.43755
Modèle 1 entraîné


In [3]:
num_labels = 5

# MODÈLE 2 MULTICLASSE - Hyperparamètres légèrement optimisés
params_multi = {
    'max_depth': 6,           # 4 → 6 (plus expressif pour 5 classes)
    'eta': 0.05,
    'gamma': 0.3,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_lambda': 8,          # 10 → 8
    'alpha': 0.1,
    'min_child_weight': 4,    # 5 → 4
    'tree_method': 'hist',
    'objective': 'multi:softprob',
    'num_class': num_labels,
    'verbosity': 1
}

counts = Counter(y_train_m)
total_samples = sum(counts.values())
sample_weights = np.array([total_samples / (len(counts) * counts[c]) for c in y_train_m])

dtrain_m = xgb.DMatrix(x_train_m, label=y_train_m, weight=sample_weights)
dtest_m  = xgb.DMatrix(x_test_m,  label=y_test_m)

evals_m = [(dtrain_m,'train'), (dtest_m,'test')]

model_multi = xgb.train(
    params_multi, dtrain_m,
    num_boost_round=500,
    evals=evals_m,
    early_stopping_rounds=30,
    verbose_eval=50
)
print("Modèle 2 entraîné")

[0]	train-mlogloss:1.49421	test-mlogloss:1.52661
[50]	train-mlogloss:0.12485	test-mlogloss:0.64582
[81]	train-mlogloss:0.03741	test-mlogloss:0.71590
Modèle 2 entraîné


In [4]:
joblib.dump(model_binary,'../models/model_binary.pkl')
joblib.dump(model_multi,'../models/model_multiclass.pkl')
shutil.copy('../data/processed/scaler.pkl','../models/scaler.pkl')
shutil.copy('../data/processed/feature_cols.pkl','../models/feature_cols.pkl')
metadata = {'class_names_binary':['Normal','Attack'],
            'class_names_5class':['Normal','DoS','Probe','R2L','U2R'],
            'family_to_id_5class':{'normal':0,'dos':1,'probe':2,'r2l':3,'u2r':4}}
joblib.dump(metadata,'../models/metadata.pkl')
print(" 5 fichiers .pkl dans models/")

 5 fichiers .pkl dans models/
